# 🧼 Hand-Wash Project: Data Exploration

This notebook provides a basic analysis of our currently annotated dataset (`outputs/annotations.json`).

In [ ]:
import os
import json
import sys
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Ensure 'src' is in path if we need custom modules
sys.path.append(os.path.abspath("../"))
from src.config import DATA_CLIPS_DIR, OUTPUTS_DIR
from src import config # Alternative access

## 📄 Loading Annotations

In [ ]:
ANNOTATIONS_PATH = OUTPUTS_DIR / "annotations.json"

with open(ANNOTATIONS_PATH, 'r', encoding='utf-8') as f:
    annotations = json.load(f)

print(f"Loaded {len(annotations)} potential clips from annotations file.")

## 📊 Basic Statistics

In [ ]:
all_events = []
for clip_name, data in annotations.items():
    if data.get("exclude", False): continue
    for event in data.get("events", []):
        duration = event["end_sec"] - event["start_sec"]
        all_events.append({
            "clip": clip_name,
            "start": event["start_sec"],
            "end": event["end_sec"],
            "duration": duration
        })

df = pd.DataFrame(all_events)

print(f"Total clips in file:      {len(annotations)}")
print(f"Active events found:     {len(df)}")
print(f"Clips with ≥ 1 event:   {len(df['clip'].unique())}")
print(f"Average wash duration:  {df['duration'].mean():.2f} sec")

df.head()

## 📈 Distribution of Wash Durations

In [ ]:
if not df.empty:
    plt.figure(figsize=(10, 5))
    plt.hist(df['duration'], bins=20, color='skyblue', edgecolor='black')
    plt.title("Distribution of Hand-Wash Durations (Annotated)")
    plt.xlabel("Duration (seconds)")
    plt.ylabel("Count")
    plt.grid(axis='y', alpha=0.3)
    plt.show()
else:
    print("No events to plot!")

## 🖼️ Visualizing a Sample Wash Event

Shows a frame from the start of a random wash event.

In [ ]:
import random

if not df.empty:
    sample = df.sample(1).iloc[0]
    clip_path = DATA_CLIPS_DIR / "unlabeled" / sample['clip']
    if not clip_path.exists():
        # Check labeled dir just in case
        clip_path = DATA_CLIPS_DIR / "labeled" / sample['clip']
        
    if clip_path.exists():
        cap = cv2.VideoCapture(str(clip_path))
        cap.set(cv2.CAP_PROP_POS_MSEC, sample['start'] * 1000)
        ret, frame = cap.read()
        if ret:
            plt.figure(figsize=(12, 7))
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.title(f"Event in {sample['clip']} starting at {sample['start']}s")
            plt.axis('off')
            plt.show()
        cap.release()
    else:
        print(f"Clip not found at path: {clip_path}")
else:
    print("No events to visualize!")